# 기본 설정

라이브러리 import

In [ ]:
# 기본 라이브러리
import os
import random
import numpy as np
import pandas as pd
import re
import torch
# 모델 및 평가 관련 라이브러리
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

# 데이터 불균형 처리 라이브러리
from imblearn.over_sampling import SMOTE, RandomOverSampler
from collections import Counter

# 하이퍼파라미터 최적화 및 전처리
import optuna
from sklearn.preprocessing import OneHotEncoder

# 경고 무시 설정
import warnings
warnings.filterwarnings("ignore")

c:\Users\user\anaconda3\envs\py_39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


라이브러리 버전 확인

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import sklearn
import imblearn
import optuna
import re
import sys

print("OS:", os.name) # 윈도우 OS 사용
print("Python version:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Imbalanced-learn:", imblearn.__version__)
print("Optuna:", optuna.__version__)
print("re:", re.__version__)

OS: nt
Python version: 3.9.15 | packaged by conda-forge | (main, Nov 22 2022, 08:39:05) [MSC v.1929 64 bit (AMD64)]
NumPy: 1.26.4
Pandas: 2.2.3
PyTorch: 2.5.1
Scikit-learn: 1.1.3
Imbalanced-learn: 0.12.4
Optuna: 4.2.1
re: 2.2.1


c:\Users\user\anaconda3\envs\py_39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


랜덤시드 고정

In [2]:
RANDOM_STATE = 736665
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(RANDOM_STATE)

seed_everything(RANDOM_STATE)

## 데이터 불러오기

In [5]:
train = pd.read_csv('./data/train.csv').drop(columns=['ID'])
test = pd.read_csv('./data/test.csv').drop(columns=['ID'])
print(train.shape, test.shape)
train.head()

(256351, 68) (90067, 67)


,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,착상 전 유전 진단 사용 여부,...,기증 배아 사용 여부,대리모 여부,PGD 시술 여부,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,임신 성공 여부
0,TRZKPL,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,0.0,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0
1,TRYBLT,만45-50세,NaN,IVF,ICSI,0,알 수 없음,0.0,NaN,0.0,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
2,TRVNRY,만18-34세,NaN,IVF,IVF,1,기록되지 않은 시행,0.0,NaN,0.0,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,2.0,NaN,0
3,TRJXFG,만35-37세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,0.0,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
4,TRVNRY,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,0.0,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0


In [6]:
train['임신 성공 여부'].value_counts()

임신 성공 여부
0    190123
1     66228
Name: count, dtype: int64

# 데이터 전처리

In [5]:
# 한 컬럼에 값이 하나이며 결측치가 없는 열 제거
del_col = []
for col in train.columns:
    if train[col].nunique() == 1:
        if train[col].isnull().sum() == 0:
            print(train[col].value_counts())
            print('결측치 개수 :', train[col].isnull().sum())
            del_col.append(col)
del_col

불임 원인 - 여성 요인
0    256351
Name: count, dtype: int64
결측치 개수 : 0


['불임 원인 - 여성 요인']

In [6]:
# 전부 결측치인 열 제거 
for col in train.columns:
    if train[col].notnull().sum() == 0:
        del_col.append(col)
del_col

['불임 원인 - 여성 요인']

In [7]:
train.drop(columns=del_col, inplace=True)
test.drop(columns=del_col, inplace=True)
train.shape, test.shape

((256351, 67), (90067, 66))

### 결측치 처리

In [8]:
temp = pd.DataFrame(train.isnull().sum(), columns=['결측치 개수'])
temp = temp[temp['결측치 개수'] > 0]
temp['결측치 비율'] = temp['결측치 개수'] / train.shape[0] * 100
temp

,결측치 개수,결측치 비율
임신 시도 또는 마지막 임신 경과 연수,246981,96.344855
특정 시술 유형,2,0.000780
단일 배아 이식 여부,6291,2.454057
착상 전 유전 검사 사용 여부,253633,98.939735
착상 전 유전 진단 사용 여부,6291,2.454057
배아 생성 주요 이유,6291,2.454057
총 생성 배아 수,6291,2.454057
미세주입된 난자 수,6291,2.454057
미세주입에서 생성된 배아 수,6291,2.454057
이식된 배아 수,6291,2.454057


In [9]:
## 결측치 처리
to_zero_columns = []
train[to_zero_columns] = train[to_zero_columns].fillna(0)
test[to_zero_columns] = test[to_zero_columns].fillna(0)


to_minus_columns = ['난자 채취 경과일']
train[to_minus_columns] = train[to_minus_columns].fillna(-1)
test[to_minus_columns] = test[to_minus_columns].fillna(-1)


to_one_columns = []
train[to_one_columns] = train[to_one_columns].fillna(1)
test[to_one_columns] = test[to_one_columns].fillna(1)


to_drop_columns = ['임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일', '배아 해동 경과일']
train = train.drop(columns=to_drop_columns)
test = test.drop(columns=to_drop_columns)

to_mean_columns = ['배아 이식 경과일', '난자 혼합 경과일']
train[to_mean_columns] = train[to_mean_columns].fillna(train[to_mean_columns].mean())
test[to_mean_columns] = test[to_mean_columns].fillna(train[to_mean_columns].mean())


train['특정 시술 유형'].fillna(train['특정 시술 유형'].value_counts().index[0], inplace=True)
test['특정 시술 유형'].fillna(train['특정 시술 유형'].value_counts().index[0], inplace=True)

train.isnull().sum().sum(), test.isnull().sum().sum()

(894338, 313512)

In [10]:
# PGD 시술 여부 또는 PGS 시술 여부가 1이면 착상 전 유전 검사 사용 여부도 1로 설정하는 함수
def update_preimplantation_testing(df):
    df.loc[(df["착상 전 유전 진단 사용 여부"] == 1), "PGD 시술 여부"] = 1
    df.loc[(df["PGD 시술 여부"] == 1), "착상 전 유전 진단 사용 여부"] = 1
    df.loc[((df["착상 전 유전 검사 사용 여부"] == 1) & (df["PGD 시술 여부"].isna())), "PGS 시술 여부"] = 1
    return df

# train 및 test 데이터프레임 전처리 적용
train = update_preimplantation_testing(train)
test = update_preimplantation_testing(test)
train.isnull().sum().sum(), test.isnull().sum().sum()

(892558, 312901)

In [11]:
temp = pd.DataFrame(train.isnull().sum(), columns=['결측치 개수'])
temp = temp[temp['결측치 개수'] > 0]
temp['결측치 비율'] = temp['결측치 개수'] / train.shape[0] * 100
temp

,결측치 개수,결측치 비율
단일 배아 이식 여부,6291,2.454057
착상 전 유전 검사 사용 여부,253633,98.939735
착상 전 유전 진단 사용 여부,6291,2.454057
배아 생성 주요 이유,6291,2.454057
총 생성 배아 수,6291,2.454057
미세주입된 난자 수,6291,2.454057
미세주입에서 생성된 배아 수,6291,2.454057
이식된 배아 수,6291,2.454057
미세주입 배아 이식 수,6291,2.454057
저장된 배아 수,6291,2.454057


In [12]:
# 시술 유형이 DI라서 생기는 결측치 따로 처리
print(temp[temp['결측치 개수'] == 6291].index)
for col in temp[temp['결측치 개수'] == 6291].index:
    if train[col].dtype == 'object':
        train[col].fillna('DI_알 수 없음', inplace=True)
        test[col].fillna('DI_알 수 없음', inplace=True)

    elif train[col].nunique() == 2:
        train[col].fillna(train[col].mode()[0], inplace=True)
        test[col].fillna(train[col].mode()[0], inplace=True)

    else:
        train[col] = train[col].fillna(train[col].mean())
        test[col] = test[col].fillna(train[col].mean())
        
train.isnull().sum().sum(), test.isnull().sum().sum()

Index(['단일 배아 이식 여부', '착상 전 유전 진단 사용 여부', '배아 생성 주요 이유', '총 생성 배아 수',
       '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수',
       '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수',
       '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수', '동결 배아 사용 여부',
       '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부'],
      dtype='object')


(760447, 267205)

In [13]:
to_drop_columns = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']
train = train.drop(columns=to_drop_columns)
test = test.drop(columns=to_drop_columns)
train.isnull().sum().sum(), test.isnull().sum().sum()

(0, 0)

### 범주형 변수 인코딩

In [ ]:
convert_dict1 = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}

# 변환할 컬럼 리스트
columns_to_convert1 = [
    "총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수", "DI 시술 횟수",
    "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수"]

# 변환 딕셔너리 생성
convert_dict2 = {'만18-34세' : 26, '만35-37세' : 36, '만38-39세' : 39, '만40-42세' : 41, '만43-44세' : 44,
                 '만45-50세' : 48, '알 수 없음' : -1}

# 변환할 컬럼 리스트
columns_to_convert2 = ["시술 당시 나이"]

convert_dict3 = {'만20세 이하' : 20, '만21-25세' : 23, '만26-30세' : 28, '만31-35세' : 33,
                 '만36-40세' : 38, '만41-45세' : 43, '알 수 없음' : -1}

# 변환할 컬럼 리스트
columns_to_convert3 = ["난자 기증자 나이", "정자 기증자 나이"]

In [14]:
train[columns_to_convert1] = train[columns_to_convert1].replace(convert_dict1)
test[columns_to_convert1] = test[columns_to_convert1].replace(convert_dict1)

train[columns_to_convert2] = train[columns_to_convert2].replace(convert_dict2)
test[columns_to_convert2] = test[columns_to_convert2].replace(convert_dict2)

train[columns_to_convert3] = train[columns_to_convert3].replace(convert_dict3)
test[columns_to_convert3] = test[columns_to_convert3].replace(convert_dict3)

In [15]:
categorical_columns = train.select_dtypes(include='object').columns
categorical_columns

Index(['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유', '난자 출처',
       '정자 출처'],
      dtype='object')

In [ ]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore')
train_encoded = train.copy()
test_encoded = test.copy()

# 원핫 인코딩 적용
train_encoded_array = onehot_encoder.fit_transform(train[categorical_columns])
test_encoded_array = onehot_encoder.transform(test[categorical_columns])

# 새로운 DataFrame으로 변환
train_encoded_df = pd.DataFrame(train_encoded_array.toarray(), columns=onehot_encoder.get_feature_names_out(categorical_columns), index=train.index) # Changed to toarray()
test_encoded_df = pd.DataFrame(test_encoded_array.toarray(), columns=onehot_encoder.get_feature_names_out(categorical_columns), index=test.index) # Changed to toarray()

# 기존 데이터에서 범주형 컬럼 제거 후 원핫 인코딩된 데이터 추가
train_encoded = train.drop(columns=categorical_columns).join(train_encoded_df)
test_encoded = test.drop(columns=categorical_columns).join(test_encoded_df)
train_encoded.shape, test_encoded.shape

((256351, 112), (90067, 111))

In [ ]:
# 컬럼명 오류 안나도록 변경
train_encoded.columns = [re.sub(r'[^\w\s]', '_', col) for col in train_encoded.columns]
test_encoded.columns = [re.sub(r'[^\w\s]', '_', col) for col in test_encoded.columns]

# 모델링

### 샘플링 실험

In [19]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 임신 성공 여부을 제외한 모든 열을 feature로 사용
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]

# train 데이터를 5개로 나누어 교차 검증
under_ratio_df = pd.DataFrame(columns=["ratio", 'fold', "model", 'auc'])

ratio = 'default'
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"{ratio}, Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    print('모델 학습 진행')
    class_counts = Counter(y_train)
    negative_count = class_counts[0]  # 음성 클래스 (보통 0)
    positive_count = class_counts[1]  # 양성 클래스 (보통 1)

    # 클래스 불균형 비율 계산
    class_imbalance_ratio = negative_count / positive_count if positive_count > 0 else 1
    import xgboost as xgb
    import lightgbm as lgb
    import catboost as cat
    xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=class_imbalance_ratio)
    lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, is_unbalance=True, verbose=-1)
    cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")
    for model in [xgb, lgbm, cat]:
        print(model.__class__.__name__)
        model.fit(X_train, y_train)

        y_proba = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_proba)

        under_ratio_df.loc[len(under_ratio_df), :] = [ratio, i+1, model.__class__.__name__, auc]

for ratio in range(1, 3):
    normal_ratio = ratio

    for i, index in enumerate(skf.split(X, y)):
        # 데이터 분할
        print(f"{ratio}, Fold {i + 1}")
        X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
        y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
        print(Counter(y_train), Counter(y_val))

        train = pd.concat([X_train, y_train], axis=1)
        val = pd.concat([X_val, y_val], axis=1)

        df_normal = train[train["임신 성공 여부"] == 0]
        df_abnormal = train[train["임신 성공 여부"] == 1]
        print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

        df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
        df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
        X_under, y_under = df_concat.drop(columns=["임신 성공 여부"]), df_concat["임신 성공 여부"]

        print("언더샘플링 후 train의 클래스 비율:", Counter(y_under))

        print('모델 학습 진행')
        class_counts = Counter(y_under)
        negative_count = class_counts[0]  # 음성 클래스 (보통 0)
        positive_count = class_counts[1]  # 양성 클래스 (보통 1)

        # 클래스 불균형 비율 계산
        class_imbalance_ratio = negative_count / positive_count if positive_count > 0 else 1
        import xgboost as xgb
        import lightgbm as lgb
        import catboost as cat
        xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=class_imbalance_ratio)
        lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, is_unbalance=True, verbose=-1)
        cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")
        for model in [xgb, lgbm, cat]:
            print(model.__class__.__name__)
            model.fit(X_under, y_under)

            y_proba = model.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
          
            under_ratio_df.loc[len(under_ratio_df), :] = [ratio, i+1, model.__class__.__name__, auc]


ratio = 'ros'
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"{ratio}, Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)
    print("Random Oversampling 클래스 비율:", Counter(y_ros))
    print('모델 학습 진행')
    class_counts = Counter(y_ros)
    negative_count = class_counts[0]  # 음성 클래스 (보통 0)
    positive_count = class_counts[1]  # 양성 클래스 (보통 1)

    # 클래스 불균형 비율 계산
    class_imbalance_ratio = negative_count / positive_count if positive_count > 0 else 1
    import xgboost as xgb
    import lightgbm as lgb
    import catboost as cat
    xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=class_imbalance_ratio)
    lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, is_unbalance=True, verbose=-1)
    cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")

    for model in [xgb, lgbm, cat]:
        print(model.__class__.__name__)
        model.fit(X_ros, y_ros)

        y_proba = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_proba)

        under_ratio_df.loc[len(under_ratio_df), :] = [ratio, i+1, model.__class__.__name__, auc]

default, Fold 1
Counter({0: 152098, 1: 52982}) Counter({0: 38025, 1: 13246})
모델 학습 진행
XGBClassifier
LGBMClassifier
CatBoostClassifier
default, Fold 2
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
모델 학습 진행
XGBClassifier
LGBMClassifier
CatBoostClassifier
default, Fold 3
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
모델 학습 진행
XGBClassifier
LGBMClassifier
CatBoostClassifier
default, Fold 4
Counter({0: 152099, 1: 52982}) Counter({0: 38024, 1: 13246})
모델 학습 진행
XGBClassifier
LGBMClassifier
CatBoostClassifier
default, Fold 5
Counter({0: 152099, 1: 52982}) Counter({0: 38024, 1: 13246})
모델 학습 진행
XGBClassifier
LGBMClassifier
CatBoostClassifier
1, Fold 1
Counter({0: 152098, 1: 52982}) Counter({0: 38025, 1: 13246})
Total: Normal: 152098, AbNormal: 52982
언더샘플링 후 train의 클래스 비율: Counter({0: 52982, 1: 52982})
모델 학습 진행
XGBClassifier
LGBMClassifier
CatBoostClassifier
1, Fold 2
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
Total: Normal: 152098, AbNormal: 52983


In [20]:
# 예측 알고리즘 별 k fold 성능
under_ratio_df.groupby(["ratio", "model"]).mean()[['auc']]

auc
ratio   model                       
1       CatBoostClassifier  0.735787
        LGBMClassifier       0.73778
        XGBClassifier       0.733022
2       CatBoostClassifier  0.736441
        LGBMClassifier       0.73852
        XGBClassifier       0.734382
default CatBoostClassifier  0.736269
        LGBMClassifier      0.738864
        XGBClassifier       0.735134
ros     CatBoostClassifier  0.733794
        LGBMClassifier      0.738255
        XGBClassifier       0.734634

In [21]:
# 샘플링 별 k fold 성능
under_ratio_df.groupby(["ratio"])[['auc']].mean().sort_values(by='auc', ascending=False)

,auc
ratio,
default,0.736756
2,0.736448
ros,0.735561
1,0.73553


In [22]:
# 성능 상위 5개 case 
top5_df = under_ratio_df.groupby(["ratio", "model"]).mean()[['auc']].sort_values(by='auc', ascending=False).head()
top5_df.reset_index(inplace=True)
top5_df

,ratio,model,auc
0,default,LGBMClassifier,0.738864
1,2,LGBMClassifier,0.73852
2,ros,LGBMClassifier,0.738255
3,1,LGBMClassifier,0.73778
4,2,CatBoostClassifier,0.736441


#### 하이퍼 파라미터 튜닝

optuna 적용

In [ ]:
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]

# Optuna objective 설정
def objective(trial, ratio, model_name):
    # k fold 교차 검증 -> optuna 자체의 cv 대신 사용 -> 샘플링 기법을 적용하기 위해서
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    auc_scores = []
    for index in skf.split(X, y):
        X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
        y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
        train = pd.concat([X_train, y_train], axis=1)

        # 샘플링 기법 적용
        if ratio == 'default':
            X_train_sample, y_train_sample = X_train, y_train

        elif ratio == 'ros':
            ros = RandomOverSampler(random_state=RANDOM_STATE)
            X_ros, y_ros = ros.fit_resample(X_train, y_train)
            X_train_sample, y_train_sample = X_ros, y_ros 

        elif ratio == 'smote':
            smote = SMOTE(sampling_strategy=0.7, random_state=RANDOM_STATE)
            X_smote, y_smote = smote.fit_resample(X_train, y_train)
            X_train_sample, y_train_sample = X_smote, y_smote

        else:
            normal_ratio = ratio
            df_normal = train[train["임신 성공 여부"] == 0]
            df_abnormal = train[train["임신 성공 여부"] == 1]

            df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
            df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
            X_train_sample, y_train_sample = df_concat.drop(columns=["임신 성공 여부"]), df_concat["임신 성공 여부"]

        # 모델 별로 탐색 파라미터 지정
        if model_name == 'XGBClassifier':
            param = {
                'objective': 'binary:logistic',
                'eval_metric': 'auc',
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),  # 학습률 범위 좁힘
                'max_depth': trial.suggest_int('max_depth', 3, 7),  # max_depth 범위 좁힘
                'min_child_weight': trial.suggest_loguniform('min_child_weight', 1e-2, 5),  # 최소 가중치 범위 좁힘
                'subsample': trial.suggest_uniform('subsample', 0.6, 0.9),  # 샘플 비율 범위 좁힘
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 0.9),  # 특성 비율 범위 좁힘
                'lambda': trial.suggest_loguniform('lambda', 1e-3, 1.0),  # L2 정규화 범위 좁힘
                'alpha': trial.suggest_loguniform('alpha', 1e-3, 1.0),  # L1 정규화 범위 좁힘
                'n_estimators': trial.suggest_int('n_estimators', 500, 5000),
                'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1, 5),  # 불균형 클래스 가중치 범위 좁힘
                'gamma': trial.suggest_uniform('gamma', 0, 3),  # 트리 분할 시 최소 손실 감소값 범위 좁힘
                'max_delta_step': trial.suggest_int('max_delta_step', 0, 5)  # 과적합 방지 범위 좁힘
            }
            model = xgb.XGBClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)

        elif model_name == 'LGBMClassifier':
            param = {
                'objective': 'binary',
                'metric': 'auc',
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.1),  # 학습률 범위 확장
                'num_leaves': trial.suggest_int('num_leaves', 20, 80),  # num_leaves 범위 확장
                'max_depth': trial.suggest_int('max_depth', 3, 10),  # max_depth 범위 확장
                'min_child_samples': trial.suggest_int('min_child_samples', 10, 120),  # 최소 샘플 수 범위 확장
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),  # 샘플링 비율 범위 확장
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),  # 특성 비율 범위 확장
                'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-4, 10),  # L1 정규화 범위 확장
                'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-4, 2.0),  # L2 정규화 범위 확장
                'max_bin': trial.suggest_int('max_bin', 100, 600),  # bin 수 범위 확장
                'n_estimators': trial.suggest_int('n_estimators', 500, 6000),  # n_estimators 범위 확장
                'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 0.5, 5),  # 클래스 불균형 조정
                'min_child_weight': trial.suggest_loguniform('min_child_weight', 1e-3, 5),  # 최소 가중치 범위 확장
                'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),  # 특성 비율 범위 확장
                'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),  # 배깅 샘플링 비율 범위 확장
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 7)  # 배깅 적용 주기 범위 확장
            }
            model = lgb.LGBMClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)
            
        elif model_name == 'CatBoostClassifier':
            param = {
                'objective': 'Logloss',
                'eval_metric': 'AUC',
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),  # 학습률 범위 좁힘
                'depth': trial.suggest_int('depth', 3, 7),  # depth 범위 좁힘
                'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-3, 5.0),  # L2 정규화 범위 좁힘
                'random_strength': trial.suggest_uniform('random_strength', 0.5, 3),  # 잡음 정도 범위 좁힘
                'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0, 0.3),  # 배깅 온도 범위 좁힘
                # 'border_count': trial.suggest_int('border_count', 32, 100),  # border 수 범위 좁힘
                'colsample_bylevel': trial.suggest_uniform('colsample_bylevel', 0.6, 0.9),  # 특성 비율 범위 좁힘
                'iterations': trial.suggest_int('iterations', 500, 2000),
                # 'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1, 5),  # 불균형 클래스 가중치 범위 좁힘
                'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),  # 리프 노드 최소 데이터 수 범위 좁힘
                'max_bin': trial.suggest_int('max_bin', 100, 150)  # bin 수 범위 좁힘
            }
            model = cat.CatBoostClassifier(**param, verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")    

        else:
            raise ValueError("Invalid model name")
        model.fit(X_train_sample, y_train_sample)

        # 예측 및 AUC 평가
        y_prob = model.predict_proba(X_val)[:, 1]  # 클래스 1의 확률 예측
        auc = roc_auc_score(y_val, y_prob)
        auc_scores.append(auc)

    # 평균 AUC 반환 (Optuna는 최대화 목표)
    return np.mean(auc_scores)

In [ ]:
# 최적화 함수 정의
def optimize_ran(ratio, model_name):
    def objective_wrapper(trial):
        return objective(trial, ratio, model_name)
    
    # **초반에는 RandomSampler 사용 (빠른 탐색)**
    study = optuna.create_study(
        direction='maximize', 
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)  # 랜덤 샘플링 시드 고정
    )
    study.optimize(objective_wrapper, n_trials=30)  # 초반 30회 랜덤 탐색

    # **이후 TPESampler로 변경 (정밀 탐색)**
    study.sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)  # TPESampler 시드 고정
    study.optimize(objective_wrapper, n_trials=100)  # 추가 100회 정밀 탐색

    print('Best hyperparameters: ', study.best_params, study.best_value)
    return study.best_params, study.best_value

# optuna 진행
params, auc, study = optimize_ran(ratio=top5_df['ratio'][0], model_name=top5_df['model'][0])
top5_df.loc[0, ['param', 'tuned_auc']] =  params, auc

[I 2025-02-24 01:48:00,501] A new study created in memory with name: no-name-3e2f1f9e-0b61-42b6-a022-c99f45d969d8


0


[I 2025-02-24 01:51:04,382] Trial 0 finished with value: 0.7341495983239276 and parameters: {'learning_rate': 0.015425143772139221, 'num_leaves': 66, 'max_depth': 9, 'min_child_samples': 106, 'subsample': 0.8653901486840079, 'colsample_bytree': 0.9983917544473779, 'lambda_l1': 0.6105295749847629, 'lambda_l2': 0.044168222863989644, 'max_bin': 126, 'n_estimators': 2273, 'scale_pos_weight': 1.95333788496361, 'min_child_weight': 0.05291362037823648, 'feature_fraction': 0.9523240386576735, 'bagging_fraction': 0.6518583351261913, 'bagging_freq': 7}. Best is trial 0 with value: 0.7341495983239276.
[I 2025-02-24 01:53:50,136] Trial 1 finished with value: 0.7339573104868691 and parameters: {'learning_rate': 0.03116375846521187, 'num_leaves': 30, 'max_depth': 10, 'min_child_samples': 46, 'subsample': 0.7221916135181802, 'colsample_bytree': 0.7974033679912669, 'lambda_l1': 0.009015009783863792, 'lambda_l2': 0.22318604907203787, 'max_bin': 309, 'n_estimators': 3278, 'scale_pos_weight': 1.158474005

Best hyperparameters:  {'learning_rate': 0.008826697606254305, 'num_leaves': 22, 'max_depth': 8, 'min_child_samples': 24, 'subsample': 0.5213248033529081, 'colsample_bytree': 0.588824074555704, 'lambda_l1': 5.592126646866256, 'lambda_l2': 0.001824157224218552, 'max_bin': 458, 'n_estimators': 1858, 'scale_pos_weight': 1.5152611515102326, 'min_child_weight': 1.6053944671635865, 'feature_fraction': 0.5339665006468401, 'bagging_fraction': 0.5627921003191587, 'bagging_freq': 3} 0.7401076532100873


In [35]:
top5_df

,ratio,model,auc,param,tuned_auc
0,default,LGBMClassifier,0.738864,"{'learning_rate': 0.008826697606254305, 'num_l...",0.740108
1,2,LGBMClassifier,0.73852,NaN,NaN
2,ros,LGBMClassifier,0.738255,NaN,NaN
3,1,LGBMClassifier,0.73778,NaN,NaN
4,2,CatBoostClassifier,0.736441,NaN,NaN


### 최종 예측

In [ ]:
# train_data에서 임신 성공 여부을 제외한 모든 열을 feature로 사용
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]

# 최적 샘플링 기법 선택
best_ratio = top5_df.loc[0, 'ratio']
print(best_ratio)
if best_ratio == 'default':
    X_sample, y_sample = X, y

elif best_ratio == 'ros':
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X, y)
    print("Random Oversampling 클래스 비율:", Counter(y_ros))
    X_sample, y_sample = X_ros, y_ros 
else:
    normal_ratio = best_ratio
    df_normal = train_encoded[train_encoded["임신 성공 여부"] == 0]
    df_abnormal = train_encoded[train_encoded["임신 성공 여부"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_sample, y_sample = df_concat.drop(columns=["임신 성공 여부"]), df_concat["임신 성공 여부"]

import xgboost as xgb
import lightgbm as lgb
import catboost as cat

# 최적 모델, 하이퍼 파라미터 선택
param = top5_df.loc[0, 'param']
if top5_df.loc[0, 'model'] == 'XGBClassifier':
    model = xgb.XGBClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)
elif top5_df.loc[0, 'model'] == 'LGBMClassifier':
    model = lgb.LGBMClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)
elif top5_df.loc[0, 'model'] == 'CatBoostClassifier':
    model =  cat.CatBoostClassifier(**param, random_state=RANDOM_STATE, auto_class_weights="Balanced")

model.fit(X_sample, y_sample)
test_proba = model.predict_proba(test_encoded)[:, 1]
test_proba

샘플링 진행
default


array([0.00255696, 0.00147716, 0.21474631, ..., 0.58934249, 0.28238356,
       0.00268898])

## 결과 저장 

In [ ]:
sample_submission = pd.read_csv('./data/sample_submission.csv')
sample_submission['probability'] = test_proba

## 후처리

[train 데이터에 대해서]
- 시술 당시 나이가 '알 수 없음'이면 항상 '임신 성공 여부'는 0의 값을 가짐
- 배아 생성 주요 이유가 '난자 저장용', '기증용' 이면 항상 '임신 성공 여부'는 0의 값을 가짐

-> 따라서, 이 경우에 해당하는 최종 예측 확률값을 0으로 지정하는 후처리 진행

In [ ]:
temp_list = ['난자 저장용',  '기증용']
sample_submission.loc[test[test['배아 생성 주요 이유'].isin(temp_list)].index, 'probability'] = 0
sample_submission.loc[test[test['시술 당시 나이'] == -1].index, 'probability'] = 0
sample_submission.to_csv('./data/최종제출본.csv', index=False)